# Death Count Extraction: LLM Rare-Bin Improvement Study

Evaluates prompt-engineering interventions (L1–L3) on the rare death-count bins (3–5 and 6+) using Llama-3.1-8B-Instruct as primary model and GPT-4o-mini as ceiling reference.

**Interventions**

| ID | Name | Description |
|---|---|---|
| L0 | Baseline | Zero-shot, no attacker-death guidance (reference) |
| L1 | Attacker deaths clarification | Adds one sentence: count all reported deaths including claimed attacker casualties |
| L2 | Bin-balanced few-shot | 5 examples (one per bin) with L1 instruction |
| L3 | Hard-case few-shot | 4 examples targeting known failure modes with L1 instruction |
| L4 | Combined few-shot | All 9 examples (L2 + L3) with L1 instruction |

**Protocol:** run each variant on the **validation set** first to check for regressions on zero/low-count cases, then on the **test set** for final results.

See `papers/death-counts/notes/rare-bin-attack-plan.md` for full study design.

## 1. Setup

### 1.1 Colab Setup

In [ ]:
# 1) Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# 2) Clone or update the repo
BRANCH = "main"  # Change if working on a feature branch
!rm -rf /content/code-satp
!git clone -b $BRANCH --depth 1 https://github.com/eteitelbaum/code-satp.git /content/code-satp

# 3) Install dependencies
%pip install -qU pip setuptools wheel
%pip install -r /content/code-satp/models/count-models/requirements.txt

# 4) Paths and sys.path
import pathlib, sys
pathlib.Path("/content/drive/MyDrive/colab/satp-results").mkdir(parents=True, exist_ok=True)
pathlib.Path("/content/drive/MyDrive/colab/satp-results/death-counts-rare-bin").mkdir(parents=True, exist_ok=True)
sys.path.append("/content/code-satp/models/count-models")

# 5) GPU check
import torch
print('=' * 60)
print('SETUP COMPLETE')
print('=' * 60)
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU — training will be very slow.')

TASK_NAME = "death-counts-rare-bin"


### 1.2 Import Libraries

In [ ]:
import os, gc, json, warnings
import numpy as np
import pandas as pd
from pathlib import Path

import torch

from utils import (
    compute_metrics, print_metrics,
    parse_fatalities,
    time_inference_call,
    load_causal,
    run_causal_batch,
    run_openai_batch,
    make_input, make_input_l1, make_input_l2, make_input_l3, make_input_l4,
    llm_already_done,
)
from utils.file_io import get_task_results_dir

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Prompt variants: maps variant name -> (prompt_fn, max_input_tokens)
# L4 uses 1024 tokens because its 9-shot fixed prompt is ~412 tokens.
# All others use 1024 too for consistency (L2/L3 truncate a small number
# of long narratives at 512; 1024 eliminates truncation for all variants).
PROMPT_VARIANTS = {
    'l0': make_input,
    'l1': make_input_l1,
    'l2': make_input_l2,
    'l3': make_input_l3,
    'l4': make_input_l4,
}
MAX_INPUT_TOKENS = 1024  # safe for all variants; eliminates truncation

print('Libraries loaded.')
print(f'Prompt variants available: {list(PROMPT_VARIANTS.keys())}')


### 1.3 API Keys

In [ ]:
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('huggingface_token')
    OPENAI_API_KEY = userdata.get('openai_api_key')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_TOKEN')
    OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')

print(f'HF token: {"found" if HF_TOKEN else "NOT FOUND"}')
print(f'OpenAI key: {"found" if OPENAI_API_KEY else "NOT FOUND"}')


## 2. Data

Load the pre-split validation and test sets. All prompt variants are evaluated on the **validation set first** to check for regressions on zero/low-count bins before touching the test set.

In [ ]:
candidate_dirs = [
    Path('/content/code-satp/models/count-models/data'),
    Path.cwd() / 'models/count-models/data',
]
data_dir = next((d for d in candidate_dirs if (d / 'test.csv').exists()), None)
if data_dir is None:
    raise FileNotFoundError('Could not locate test.csv')

val_df  = pd.read_csv(data_dir / 'val.csv')
test_df = pd.read_csv(data_dir / 'test.csv')

# Ensure ID column
ID_COL = 'incident_number'
for split_name, sdf in [('val', val_df), ('test', test_df)]:
    if ID_COL not in sdf.columns:
        raise ValueError(f'{ID_COL} missing from {split_name} split')

# Bin labels for reporting
def assign_bin(n):
    if n == 0:   return '0'
    if n == 1:   return '1'
    if n == 2:   return '2'
    if n <= 5:   return '3-5'
    return '6+'

for sdf in [val_df, test_df]:
    sdf['bin'] = sdf['total_fatalities'].apply(assign_bin)

print('Validation set:')
print(val_df['bin'].value_counts().sort_index())
print('\nTest set:')
print(test_df['bin'].value_counts().sort_index())


## 3. Output Directory

In [ ]:
results_dir = get_task_results_dir(TASK_NAME, create=True)
OUTPUT_DIR = results_dir
print(f'Output directory: {OUTPUT_DIR}')

def run_and_save(name, outputs, df_input):
    """Parse outputs, compute overall + bin-level metrics, save CSV."""
    parsed = [parse_fatalities(s) for s in outputs]
    true   = df_input['total_fatalities'].values
    ok     = [bool(s and s.strip()) for s in outputs]
    metrics = compute_metrics(parsed, true, ok)
    print_metrics(metrics, name)

    # Bin-level MAE and exact match
    res = df_input[[ID_COL, 'incident_summary', 'total_fatalities', 'bin']].copy()
    res['prediction'] = parsed
    res['raw_output'] = outputs
    res['error'] = res['prediction'] - res['total_fatalities']

    print('\nBin-level metrics:')
    print(f'  {"Bin":<6}  {"N":>4}  {"MAE":>6}  {"Exact%":>7}')
    for b in ['0', '1', '2', '3-5', '6+']:
        sub = res[res['bin'] == b]
        if len(sub) == 0:
            continue
        mae   = sub['error'].abs().mean()
        exact = (sub['prediction'] == sub['total_fatalities']).mean() * 100
        print(f'  {b:<6}  {len(sub):>4}  {mae:>6.3f}  {exact:>6.1f}%')

    out_path = OUTPUT_DIR / f'{name}.csv'
    res.to_csv(out_path, index=False)
    metrics_path = OUTPUT_DIR / f'{name}_metrics.json'
    with open(metrics_path, 'w') as f:
        json.dump(metrics, f, indent=2)
    print(f'Saved: {out_path}')
    return res, metrics


## 4. Validation Runs

Run all variants on the validation set. Check that zero-death exact match does not degrade before proceeding to the test set.

GPT-4o-mini is run on L1 only (per plan) as a ceiling reference.

### 4.1 Llama-3.1-8B — all variants (val set)

In [ ]:
LLAMA_MODEL = 'meta-llama/Meta-Llama-3.1-8B-Instruct'

# Load model once, run all variants
tok, mdl = load_causal(LLAMA_MODEL, token=HF_TOKEN)
texts_val = val_df['incident_summary'].tolist()

for variant, pfn in PROMPT_VARIANTS.items():
    name = f'llama3_8b_{variant}_val'
    if llm_already_done(name, OUTPUT_DIR):
        print(f'Already done: {name}')
        continue
    print(f'\n{"="*60}')
    print(f'Running llama3_8b variant={variant} on val set')
    print(f'{"="*60}')
    outs, timing = time_inference_call(
        run_causal_batch, tok, mdl, texts_val,
        max_new_tokens=48, max_input_tokens=MAX_INPUT_TOKENS, prompt_fn=pfn
    )
    run_and_save(name, outs, val_df)

del tok, mdl
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Llama val runs complete.')


### 4.2 GPT-4o-mini — L1 only (val set, ceiling reference)

In [ ]:
name = 'gpt4o_mini_l1_val'
if not llm_already_done(name, OUTPUT_DIR):
    if not OPENAI_API_KEY:
        print('OpenAI key not found — skipping GPT-4o-mini')
    else:
        outs, timing = time_inference_call(
            run_openai_batch,
            texts_val,
            api_key=OPENAI_API_KEY,
            model_name='gpt-4o-mini',
            max_tokens=256,
            rate_limit_delay=0.1,
            prompt_fn=make_input_l1,
        )
        run_and_save(name, outs, val_df)
else:
    print(f'Already done: {name}')


### 4.3 Regression check

Verify that zero-death exact match does not degrade across variants before proceeding to the test set.

In [ ]:
print('Validation regression check — bin 0 exact match by variant')
print(f'{"Variant":<25}  {"Bin-0 exact%":>12}  {"Bin 3-5 exact%":>14}  {"Bin 6+ exact%":>13}')

for variant in PROMPT_VARIANTS:
    fname = OUTPUT_DIR / f'llama3_8b_{variant}_val.csv'
    if not fname.exists():
        continue
    df_r = pd.read_csv(fname)
    for b, col in [('0', 'Bin-0 exact%'), ('3-5', 'Bin 3-5 exact%'), ('6+', 'Bin 6+ exact%')]:
        sub = df_r[df_r['bin'] == b]
        pct = (sub['prediction'] == sub['total_fatalities']).mean() * 100 if len(sub) else float('nan')
    # recompute all three for printing
    def exact_pct(b):
        sub = df_r[df_r['bin'] == b]
        return (sub['prediction'] == sub['total_fatalities']).mean() * 100 if len(sub) else float('nan')
    print(f'  llama3_8b_{variant:<15}  {exact_pct("0"):>12.1f}  {exact_pct("3-5"):>14.1f}  {exact_pct("6+"):>13.1f}')


## 5. Test Set Runs

Run validated variants on the test set. Only proceed here after confirming no regressions in Section 4.3.

### 5.1 Llama-3.1-8B — all variants (test set)

In [ ]:
tok, mdl = load_causal(LLAMA_MODEL, token=HF_TOKEN)
texts_test = test_df['incident_summary'].tolist()

for variant, pfn in PROMPT_VARIANTS.items():
    name = f'llama3_8b_{variant}_test'
    if llm_already_done(name, OUTPUT_DIR):
        print(f'Already done: {name}')
        continue
    print(f'\n{"="*60}')
    print(f'Running llama3_8b variant={variant} on test set')
    print(f'{"="*60}')
    outs, timing = time_inference_call(
        run_causal_batch, tok, mdl, texts_test,
        max_new_tokens=48, max_input_tokens=MAX_INPUT_TOKENS, prompt_fn=pfn
    )
    run_and_save(name, outs, test_df)

del tok, mdl
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Llama test runs complete.')


### 5.2 GPT-4o-mini — L1 only (test set, ceiling reference)

In [ ]:
name = 'gpt4o_mini_l1_test'
if not llm_already_done(name, OUTPUT_DIR):
    if not OPENAI_API_KEY:
        print('OpenAI key not found — skipping GPT-4o-mini')
    else:
        outs, timing = time_inference_call(
            run_openai_batch,
            texts_test,
            api_key=OPENAI_API_KEY,
            model_name='gpt-4o-mini',
            max_tokens=256,
            rate_limit_delay=0.1,
            prompt_fn=make_input_l1,
        )
        run_and_save(name, outs, test_df)
else:
    print(f'Already done: {name}')


## 6. Results Table

Compile the bin-level metrics table from the test-set runs. Format matches the paper's Table structure (rare-bin-attack-plan.md §Results Table Structure).

In [ ]:
import pandas as pd

rows = []
run_configs = [
    # Original GPT-4o-mini L0 baseline from the main experiment (no intervention)
    ('gpt4o_mini_l0_baseline', 'GPT-4o-mini',  'L0 Baseline (original experiment)'),
    # Llama variants
    ('llama3_8b_l0_test',      'Llama-3.1-8B', 'L0 Baseline'),
    ('llama3_8b_l1_test',      'Llama-3.1-8B', 'L1 Attacker deaths clarification'),
    ('llama3_8b_l2_test',      'Llama-3.1-8B', 'L2 Bin-balanced few-shot'),
    ('llama3_8b_l3_test',      'Llama-3.1-8B', 'L3 Hard-case few-shot'),
    ('llama3_8b_l4_test',      'Llama-3.1-8B', 'L4 Combined few-shot (L2+L3)'),
    # GPT-4o-mini ceiling reference
    ('gpt4o_mini_l1_test',     'GPT-4o-mini',  'L1 (ceiling reference)'),
]

# Load original GPT-4o-mini baseline from the main experiment results
baseline_dirs = [
    Path('/content/drive/MyDrive/colab/satp-results/death-counts'),
    Path('/content/code-satp/papers/death-counts/results/death-counts-llms'),
    Path.cwd() / 'papers/death-counts/results/death-counts-llms',
]

def load_baseline_gpt():
    for d in baseline_dirs:
        p = d / 'gpt4o_mini.csv'
        if p.exists():
            df = pd.read_csv(p)
            # Normalise column names to match current format
            if 'gpt4o_mini_prediction' in df.columns:
                df = df.rename(columns={'gpt4o_mini_prediction': 'prediction',
                                        'true_label': 'total_fatalities'})
            df['bin'] = df['total_fatalities'].apply(assign_bin)
            df['error'] = df['prediction'] - df['total_fatalities']
            return df
    return None

gpt_baseline = load_baseline_gpt()

for fname, model, strategy in run_configs:
    if fname == 'gpt4o_mini_l0_baseline':
        df_r = gpt_baseline
        if df_r is None:
            print('GPT-4o-mini baseline not found — skipping')
            continue
    else:
        fpath = OUTPUT_DIR / f'{fname}.csv'
        if not fpath.exists():
            print(f'Missing: {fpath}')
            continue
        df_r = pd.read_csv(fpath)

    def mae(b):
        sub = df_r[df_r['bin'] == b]
        return sub['error'].abs().mean() if len(sub) else float('nan')
    def exact(b):
        sub = df_r[df_r['bin'] == b]
        return (sub['prediction'] == sub['total_fatalities']).mean() * 100 if len(sub) else float('nan')
    overall_mae = df_r['error'].abs().mean()
    nonzero_mae = df_r[df_r['total_fatalities'] > 0]['error'].abs().mean()
    rows.append({
        'Model': model,
        'Strategy': strategy,
        'Overall MAE': round(overall_mae, 3),
        'Nonzero MAE': round(nonzero_mae, 3),
        'Bin 3-5 MAE': round(mae('3-5'), 3),
        'Bin 6+ MAE': round(mae('6+'), 3),
        'Exact 3-5 (%)': round(exact('3-5'), 1),
        'Exact 6+ (%)': round(exact('6+'), 1),
    })

results_table = pd.DataFrame(rows)
print(results_table.to_string(index=False))

results_table.to_csv(OUTPUT_DIR / 'llm_rare_bin_results.csv', index=False)
print(f'\nSaved: {OUTPUT_DIR}/llm_rare_bin_results.csv')
